# SemanticKITTI Inference & Visualization
This notebook will run inference on a validation frame using your fine-tuned PointNet++ model and visualize the results.

In [ ]:
%%writefile test_semantickitti.py
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from semantickitti_dataset import SemanticKITTIDataset
from outdoor_pointnet import get_outdoor_model

def get_semantickitti_colors():
    return {
        0: [0, 0, 0],         1: [100, 150, 245],   2: [100, 230, 245],   3: [30, 60, 150],     
        4: [80, 30, 180],     5: [100, 80, 250],    6: [255, 30, 30],     7: [255, 40, 200],    
        8: [150, 30, 90],     9: [255, 0, 255],     10: [255, 150, 255],  11: [75, 0, 75],      
        12: [175, 0, 75],     13: [255, 200, 0],    14: [255, 120, 50],   15: [0, 175, 0],      
        16: [135, 60, 0],     17: [150, 240, 80],   18: [255, 240, 150],  19: [255, 0, 0]
    }

def main():
    DATASET_PATH = '/content/dataset_trimmed/sequences'
    # Try epoch 10 first, fallback to epoch 1 if it doesn't exist
    CHECKPOINT_PATH_10 = '/content/drive/MyDrive/checkpoints/semantickitti_epoch_10.pth'
    CHECKPOINT_PATH_1 = '/content/drive/MyDrive/checkpoints/semantickitti_epoch_1.pth'
    
    print("Loading test frame from SemanticKITTI...")
    dataset = SemanticKITTIDataset(DATASET_PATH, sequences=['00'], num_points=4096, split='val')
    point_features, point_labels = dataset[0]
    
    inputs = point_features.unsqueeze(0)
    labels = point_labels.numpy()
    xyz = inputs[0, :3, :].transpose(0, 1).numpy()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = get_outdoor_model(num_classes=20, input_channels=1)
    
    if os.path.exists(CHECKPOINT_PATH_10):
        print(f"Loading weights from {CHECKPOINT_PATH_10}...")
        model.load_state_dict(torch.load(CHECKPOINT_PATH_10, map_location=device, weights_only=False))
    elif os.path.exists(CHECKPOINT_PATH_1):
        print(f"Loading weights from {CHECKPOINT_PATH_1}...")
        model.load_state_dict(torch.load(CHECKPOINT_PATH_1, map_location=device, weights_only=False))
    else:
        print("ERROR: Could not find epoch 1 or epoch 10 checkpoint in your Google Drive!")
        return
        
    model = model.to(device)
    model.eval()
    
    print("Running inference...")
    inputs = inputs.to(device)
    with torch.no_grad():
        predictions, _ = model(inputs)
    
    pred_labels = torch.argmax(predictions, dim=2).squeeze(0).cpu().numpy()
    
    print("Plotting results...")
    color_map = get_semantickitti_colors()
    gt_colors = np.array([color_map[l] for l in labels]) / 255.0
    pred_colors = np.array([color_map[l] for l in pred_labels]) / 255.0
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    ax1.scatter(xyz[:, 0], xyz[:, 1], c=gt_colors, s=5, alpha=0.8)
    ax1.set_title("Ground Truth", fontsize=16)
    ax1.axis('equal')
    ax1.set_facecolor('black') 
    
    ax2.scatter(xyz[:, 0], xyz[:, 1], c=pred_colors, s=5, alpha=0.8)
    ax2.set_title("PointNet++ Prediction", fontsize=16)
    ax2.axis('equal')
    ax2.set_facecolor('black')
    
    plt.tight_layout()
    plt.show()

if __name__ == '__main__':
    main()


In [ ]:
!python test_semantickitti.py